## Execução local
Selecione o kernel **Python (FASE7)** e execute em ordem. Consulte `README_LOCAL.md` para instalação. Redes neurais usam PyTorch/CUDA; operações NumPy, áudio e modelos scikit-learn usam CPU.

In [ ]:
from pathlib import Path
import os, sys
# Localiza FASE7 tanto a partir do projeto quanto da pasta do notebook.
base = Path.cwd().resolve()
FASE7 = next((p for a in (base, *base.parents) for p in (a, a / "FASE7", a / "ANO1" / "FASE7") if p.name == "FASE7" and (p / "RNN").is_dir()), None)
if FASE7 is None:
    raise FileNotFoundError("Abra o Jupyter dentro da pasta FIAP ou FASE7.")
sys.path.insert(0, str(FASE7))
os.chdir(FASE7 / 'Cap 02')
print("Pasta de trabalho:", Path.cwd())


FIAP

# Redes Neurais

# Aula - Classificação de texto

Este trabalho tenta detectar spam ou ham.

# PNL

O processamento de linguagem natural (PLN) é um subcampo da linguística, ciência da computação e inteligência artificial relacionado às interações entre computadores e linguagem humana, em particular como programar computadores para processar e analisar grandes quantidades de dados de linguagem natural.

Atualmente, o PNL é o foco de interesse significativo na comunidade de aprendizado de máquina. Alguns dos casos de uso para PNL estão listados aqui:
* Chatbots
* Pesquisa (texto e áudio)
* Classificação de texto
* Análise de sentimento
* Sistema de recomendação
* Resposta a perguntas
* Reconhecimento de fala
* NLU (Entendimento da linguagem natural)
* NLG (Geração de linguagem natural)

Você encontra muitos desses casos de uso na vida cotidiana: quando visita páginas da web ou realiza uma pesquisa online de livros ou recomendações sobre filmes.

# Técnicas de PNL

A abordagem mais antiga para resolver tarefas de PNL envolve abordagens baseadas em regras, que dominaram a indústria por décadas. Exemplos de técnicas que usam abordagens baseadas em regras incluem Expressões regulares (RegExs) e Gramáticas livres de contexto (CFGs). Às vezes, as RegExs são usadas para remover tags HTML de texto que foi raspado de uma página da web ou caracteres especiais indesejados de um documento.

A segunda abordagem envolveu o treinamento de um modelo de aprendizado de máquina com alguns dados baseados em alguns recursos definidos pelo usuário. Essa técnica requer uma quantidade considerável de engenharia de recursos (uma tarefa não trivial) e inclui a análise do texto para remover conteúdo indesejado e supérfluo (incluindo palavras irrelevantes), bem como a transformação da palavra (por exemplo, conversão de maiúsculas para minúsculas).

A abordagem mais recente envolve aprendizado profundo, por meio do qual uma rede neural aprende os recursos em vez de depender de humanos para executar a engenharia de recursos. Uma das principais ideias envolve o mapeamento de palavras para números, o que nos permite mapear frases para vetores de números. Após transformar documentos em vetores, podemos executar uma miríade de operações nesses vetores. Por exemplo, usamos a noção de espaço vetorial para definir o modelo de espaço vetorial, onde a distância entre dois vetores pode ser medida pelo ângulo entre eles (relacionado à similaridade de cosseno). Se dois vetores são próximos um do outro, então é mais provável que a frase correspondente tenha significado semelhante. Sua similaridade é baseada na hipótese distribucional, que afirma que palavras nos mesmos contextos tendem a ter significado semelhante. Os modelos de PNL que usam aprendizado profundo podem compreender CNNs, RNNs, LSTMs e LSTMs bidirecionais.

# Exemplos de classificação de texto

**O que é classificação de texto?**

A classificação de texto é o processo de categorizar texto em grupos organizados. Ao usar o Processamento de Linguagem Natural (PLN), os classificadores de texto podem analisar o texto automaticamente e, em seguida, atribuir um conjunto de tags ou categorias predefinidas com base em seu conteúdo. Os classificadores de texto com PNL provaram ser uma ótima alternativa para estruturar dados textuais de forma rápida, econômica e escalável. A classificação de texto está se tornando uma parte cada vez mais importante dos negócios, pois permite obter insights facilmente dos dados e automatizar processos de negócios. Alguns dos exemplos e casos de uso mais comuns para classificação automática de texto incluem o seguinte:

* **Análise de sentimento**: o processo de entender se um determinado texto está falando positiva ou negativamente sobre um determinado assunto (por exemplo, para fins de monitoramento de marca).

* **Detecção de spam**: detecta e-mails não solicitados, indesejados e infestados de vírus (chamados de spam) e impede que eles cheguem às caixas de entrada de e-mail. Os provedores de serviços de Internet (ISPs) usam filtros de spam para garantir que não estejam distribuindo spam.

* **Detecção de Tópico**: a tarefa de identificar o tema ou tópico de um pedaço de texto (por exemplo, saber se uma análise de produto é sobre Facilidade de Uso, Suporte ao Cliente ou Preço ao analisar o feedback do cliente).

* **Detecção de Idioma**: o procedimento de detectar o idioma de um determinado texto (por exemplo, saber se um tíquete de suporte recebido está escrito em inglês ou espanhol para encaminhar automaticamente os tíquetes para a equipe apropriada).

## Instalação de bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
import string
import re
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Carregamento dos dados


In [ ]:
df = pd.read_csv('SPAM text message 20170820 - Data.csv')
print(df.shape)

## Data exploration


In [ ]:
# Inspecionando os tipos de dados
print(df.info())

# Analisando a distribuição das classes
print()
print(df['Category'].value_counts())
print()

# Visualizando a distribuição de spam/ham
df['Category'].value_counts().plot(kind='bar')
plt.title('Distribuição de Spam/Ham')
plt.xlabel('Categoria')
plt.ylabel('Contagem')
plt.show()

In [ ]:
# Explore the text data
print("Sample Spam Messages:")
print(df[df['Category'] == 'spam']['Message'].sample(5))

print("\nSample Ham Messages:")
print(df[df['Category'] == 'ham']['Message'].sample(5))

## Data cleaning


In [ ]:
def clean_text(text):
  """Limpa o texto de entrada removendo HTML, pontuação e convertendo para minúsculas."""
  if isinstance(text, str):
    text = BeautifulSoup(text, "html.parser").text
    text = re.sub(r'\|\|\|', r' ', text)
    text = re.sub(r'http\S+', r'<URL>', text)
    text = text.lower()
    text = text.replace('x', '')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
  return text

# Aplique a função clean_text à coluna 'Message' e crie uma nova coluna 'Cleaned_Message'
df['Cleaned_Message'] = df['Message'].apply(clean_text)

# Verifique se há valores ausentes na coluna 'Cleaned_Message'
print(df['Cleaned_Message'].isnull().sum())

## Data preparation


In [ ]:
# Divida os dados em conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Message'],
    df['Category'],
    test_size=0.2,
    random_state=42,
    stratify=df['Category']
)

# Preparar rótulos de destino (converter 'Categoria' em representação numérica)
label_mapping = {'ham': 0, 'spam': 1}
y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

## Feature engineering

Criando representações dos textos com Bag-of-Words (BoW), TF-IDF, e Word Embeddings.


In [ ]:
# Bag-of-Words (BoW)
vectorizer_bow = CountVectorizer()
X_train_bow = vectorizer_bow.fit_transform(X_train)
X_test_bow = vectorizer_bow.transform(X_test)


# TF-IDF
vectorizer_tfidf = TfidfVectorizer()
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)


# Representação densa aprendida SOMENTE nos textos de treino.
# NMF mantém valores não negativos, permitindo comparar também MultinomialNB.
# Diferente do placeholder original, estas features não são vetores de zeros.
from sklearn.decomposition import NMF
embedding_model = NMF(n_components=64, init="nndsvda", random_state=42, max_iter=300)
X_train_embeddings = embedding_model.fit_transform(X_train_tfidf)
X_test_embeddings = embedding_model.transform(X_test_tfidf)


## Model training



In [ ]:
# Modelos para representação BoW
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)

svm_bow = SVC()
svm_bow.fit(X_train_bow, y_train)

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

rf_bow = RandomForestClassifier()
rf_bow.fit(X_train_bow, y_train)


# Modelos para representação TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

svm_tfidf = SVC()
svm_tfidf.fit(X_train_tfidf, y_train)

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

rf_tfidf = RandomForestClassifier()
rf_tfidf.fit(X_train_tfidf, y_train)


# Modelos para Word Embeddings
lr_embeddings = LogisticRegression(max_iter=1000)
lr_embeddings.fit(X_train_embeddings, y_train)

svm_embeddings = SVC()
svm_embeddings.fit(X_train_embeddings, y_train)

rf_embeddings = RandomForestClassifier()
rf_embeddings.fit(X_train_embeddings, y_train)

nb_embeddings = MultinomialNB()
nb_embeddings.fit(X_train_embeddings, y_train)

# Dicionário de modelos treinados
trained_models = {
    'lr_bow': lr_bow,
    'svm_bow': svm_bow,
    'nb_bow': nb_bow,
    'rf_bow': rf_bow,
    'lr_tfidf': lr_tfidf,
    'svm_tfidf': svm_tfidf,
    'nb_tfidf': nb_tfidf,
    'rf_tfidf': rf_tfidf,
    'lr_embeddings': lr_embeddings,
    'svm_embeddings': svm_embeddings,
    'rf_embeddings': rf_embeddings,
    'nb_embeddings': nb_embeddings,
}

## Model evaluation


In [ ]:
evaluation_results = {}

for model_name, model in trained_models.items():
  if 'bow' in model_name:
    y_pred = model.predict(X_test_bow)
  elif 'tfidf' in model_name:
    y_pred = model.predict(X_test_tfidf)
  elif 'embeddings' in model_name:
    y_pred = model.predict(X_test_embeddings)
  else:
    continue

  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  conf_matrix = confusion_matrix(y_test, y_pred)

  evaluation_results[model_name] = {
      'accuracy': accuracy,
      'f1_score': f1,
      'confusion_matrix': conf_matrix,
  }

# Vizualiza os resultados como um dataframe ordenado por acurácia
pd.DataFrame(evaluation_results).T.sort_values(by='accuracy', ascending=False)

In [ ]:
best_model = max(evaluation_results, key=lambda k: evaluation_results[k]["accuracy"])
best_f1_model = max(evaluation_results, key=lambda k: evaluation_results[k]["f1_score"])
print("Melhor acurácia:", best_model, evaluation_results[best_model]["accuracy"])
print("Melhor F1:", best_f1_model, evaluation_results[best_f1_model]["f1_score"])


## Data visualization


In [ ]:
# Escolhendo o MELHOR e o PIOR modelo para plotar a matriz de confusão
melhor = "lr_bow"
pior = "svm_embeddings"

def plot_confusion_matrix(results, model_name):
  conf_matrix = results['confusion_matrix']
  plt.figure(figsize=(6, 4))
  sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
              xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.title(f'Confusion Matrix for {model_name}')
  plt.show()


plot_confusion_matrix(evaluation_results[melhor], melhor)
print()
plot_confusion_matrix(evaluation_results[pior], pior)